In [3]:
# Import modules.
# ---------------------------------------------------------
import pyvisa as visa
import struct
import math
import time 
import numpy as np

# Global variables
# (Modify the following global variables according to the model).
# ---------------------------------------------------------
SDS_RSC = 'USB0::0xF4EC::0x1017::SDS08A0Q808362::INSTR'
CHANNEL = "C1"
HORI_NUM = 10
tdiv_enum = [200e-12,500e-12, 1e-9,\
 2e-9, 5e-9, 10e-9, 20e-9, 50e-9, 100e-9, 200e-9, 500e-9, \
 1e-6, 2e-6, 5e-6, 10e-6, 20e-6, 50e-6, 100e-6, 200e-6, 500e-6, \
 1e-3, 2e-3, 5e-3, 10e-3, 20e-3, 50e-3, 100e-3, 200e-3, 500e-3, \
 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]

# =========================================================
# main_desc:Analyzing waveform parameters from data blocks
# =========================================================
def main_desc(recv):
    WAVE_ARRAY_1 = recv[0x3c:0x3f + 1]
    wave_array_count = recv[0x74:0x77 + 1]
    first_point = recv[0x84:0x87 + 1]
    sp = recv[0x88:0x8b + 1]
    v_scale = recv[0x9c:0x9f + 1]
    v_offset = recv[0xa0:0xa3 + 1]
    interval = recv[0xb0:0xb3 + 1]
    code_per_div = recv[0xa4:0Xa7 + 1]
    adc_bit = recv[0xac:0Xad + 1]
    delay = recv[0xb4:0xbb + 1]
    tdiv = recv[0x144:0x145 + 1]
    probe = recv[0x148:0x14b + 1]
    data_bytes = struct.unpack('i', WAVE_ARRAY_1)[0]
    point_num = struct.unpack('i', wave_array_count)[0]
    fp = struct.unpack('i', first_point)[0]
    sp = struct.unpack('i', sp)[0]
    interval = struct.unpack('f', interval)[0]
    delay = struct.unpack('d', delay)[0]
    tdiv_index = struct.unpack('h', tdiv)[0]
    probe = struct.unpack('f', probe)[0]
    vdiv = struct.unpack('f', v_scale)[0] * probe
    offset = struct.unpack('f', v_offset)[0] * probe
    code = struct.unpack('f', code_per_div)[0]
    adc_bit = struct.unpack('h', adc_bit)[0]
    tdiv = tdiv_enum[tdiv_index]
    return vdiv, offset, interval, delay, tdiv, code, adc_bit


# =========================================================
# Main program:
# =========================================================
def get_waveform():
    _rm = visa.ResourceManager()
    sds = _rm.open_resource(SDS_RSC)
    sds.timeout = 2000 # default value is 2000(2s)
    sds.chunk_size = 20 * 1024 * 1024 # default value is 20*1024(20k bytes)
    # Get the channel waveform parameter data blocks and parse them
    sds.write(":WAVeform:STARt 0")
    sds.write("WAV:SOUR {}".format(CHANNEL))
    sds.write("WAV:PREamble?")
    t0 = time.time()
    recv_all = sds.read_raw()
    print("time to read preamble: %.1f seconds" % (time.time()-t0))
    recv = recv_all[recv_all.find(b'#') + 11:]
    print(len(recv))
    vdiv, ofst, interval, trdl, tdiv, vcode_per, adc_bit = main_desc(recv)
    print(vdiv, ofst, interval, trdl, tdiv,vcode_per,adc_bit)
    # Get the waveform points and confirm the number of waveform slice reads
    points = float(sds.query(":ACQuire:POINts?").strip())
    one_piece_num = float(sds.query(":WAVeform:MAXPoint?").strip())
    read_times = math.ceil(points / one_piece_num)
    #Set the number of read points per slice, if the waveform points is greater than the maximum
    # number of slice reads
    if points > one_piece_num:
        sds.write(":WAVeform:POINt {}".format(one_piece_num))
        # Choose the format of the data returned
        sds.write(":WAVeform:WIDTh BYTE")
    if adc_bit > 8:
        sds.write(":WAVeform:WIDTh WORD")
    #Get the waveform data for each slice
    recv_byte = b''
    print("read_times", read_times)
    for i in range(0, read_times):
        start = i * one_piece_num
        #Set the starting point of each slice
        sds.write(":WAVeform:STARt {}".format(start))
        #Get the waveform data of each slice
        t0 = time.time()
        sds.write("WAV:DATA?")
        recv_rtn = sds.read_raw()
        print("read time block %d time %.2f seconds" % (i, (time.time()-t0)))
        # We need to get rid of the two "\n\n" at the end of the received bytes
        # However, we should not use rstrip since it's behavior is not so well defined for 
        # binary streams. But fortunately, there are always two, so we can just get rid of them
        # using slicing. 
        #recv_rtn = recv_rtn.rstrip()
        recv_rtn = recv_rtn[:-2]
        #Splice each waveform data based on data block information
        block_start = recv_rtn.find(b'#')
        data_digit = int(recv_rtn[block_start + 1:block_start + 2])
        data_start = block_start + 2 + data_digit
        recv_byte += recv_rtn[data_start:]
    # Unpack signed byte data.
    if adc_bit > 8:
        print("points", points)
        convert_data = struct.unpack("=%dh"%points, recv_byte)
    else:
        convert_data = struct.unpack("%db"%points, recv_byte)
    convert_data = np.array(convert_data)
    #Calculate the voltage value and time value
    time_value = []
    volt_value = []
    N = len(convert_data)
    i = np.linspace(0, N-1, N)
    volt_value = convert_data / vcode_per * float(vdiv) -float(ofst)
    time_data = float(tdiv)*HORI_NUM/2 + i*interval + float(trdl)
    return time_data, volt_value

In [4]:
import bokeh
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook
output_notebook()

Loading BokehJS ...

In [16]:
t,v = get_waveform()
x = t-t[0]
y = v
p = figure(height=200, width=600) 
p.sizing_mode = "scale_width"
#l=p.dot(x,y)
l=p.line(x,y)
target = show(p, notebook_handle=True)

time to read preamble: 0.0 seconds
347
0.0010000000474974513 0.00041666667675599456 1.9999999494757503e-05 0.0016 0.02 7680.0 16
read_times 1
read time block 0 time 0.19 seconds
points 10000.0


In [9]:
t,v = get_waveform()
l.data_source.data = dict(x=t, y=v)
push_notebook(handle=target)

time to read preamble: 0.0 seconds
347
0.0010000000474974513 0.00041666667675599456 1.9999999949504854e-06 0.0016 0.02 7680.0 16
read_times 1
read time block 0 time 0.21 seconds
points 100000.0


In [2]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook
output_notebook()

plot = figure(width=300, height=300)
plot.circle(x=[1, 2, 3], y=[1, 2, 3], radius=0.2)

show(plot)

Loading BokehJS ...